# MindLens — Model 2: Emotion Classifier (RoBERTa · dair-ai/emotion · Kaggle T4)

## Before running
1. **GPU**: Runtime → Change runtime type → **GPU T4 x2**
2. **Kaggle Secret**: Add-ons → Secrets → `HF_TOKEN` (your HuggingFace *write* token from https://huggingface.co/settings/tokens)
3. **Run All**

Trains in ~10-15 min. Pushes trained model + model card to `huggingface.co/AmiruMallawarachchi/mindlens-emotion-classifier`


In [ ]:
!pip install -q "transformers>=4.46" "datasets>=2.20" \
    "accelerate>=0.34" evaluate scikit-learn seaborn huggingface_hub


In [ ]:
from kaggle_secrets import UserSecretsClient as _USC
import os as _os
_s = _USC()
_os.environ["HF_TOKEN"] = _s.get_secret("HF_TOKEN")

import os, random, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback,
    DataCollatorWithPadding
)
from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix, classification_report
)
from huggingface_hub import HfApi, create_repo

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

MODEL_NAME   = "roberta-base"
HF_REPO_ID   = "AmiruMallawarachchi/mindlens-emotion-classifier"
HF_TOKEN     = os.environ.get("HF_TOKEN", "")
ACCURACY_GATE = 0.80

# dair-ai/emotion labels (fixed order from dataset card)
LABELS   = ["sadness", "joy", "love", "anger", "fear", "surprise"]
id2label = {i: l for i, l in enumerate(LABELS)}
label2id = {l: i for i, l in enumerate(LABELS)}


In [ ]:
ds = load_dataset("dair-ai/emotion", "split")
print(ds)
print(ds["train"][0])
# Columns: text (str), label (int 0-5)


In [ ]:
df_train = ds["train"].to_pandas()
df_val   = ds["validation"].to_pandas()
df_test  = ds["test"].to_pandas()

print(f"Train: {len(df_train):,}  Val: {len(df_val):,}  Test: {len(df_test):,}")

# 4.1 Class distribution
counts = df_train["label"].value_counts().sort_index()
counts.index = LABELS
plt.figure(figsize=(7, 4))
sns.barplot(x=counts.index, y=counts.values)
plt.title("dair-ai/emotion — Train Class Distribution")
plt.ylabel("Count"); plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig("eda_model2_class_dist.png", dpi=150, bbox_inches="tight")
plt.show()
print("Imbalance ratio:", round(counts.max() / counts.min(), 2), ": 1")

# 4.2 Token length distribution
tok_probe = AutoTokenizer.from_pretrained(MODEL_NAME)
lens = df_train["text"].astype(str).apply(
    lambda t: len(tok_probe.encode(t))
)
plt.figure(figsize=(6, 4))
sns.histplot(lens, bins=40)
p95 = int(lens.quantile(0.95))
plt.axvline(p95, color="red", linestyle="--", label=f"95th pct = {p95}")
plt.title("Token Length Distribution (train)"); plt.legend()
plt.savefig("eda_model2_token_len.png", dpi=150, bbox_inches="tight")
plt.show()
MAX_LENGTH = min(128, p95 + 8)
print(f"MAX_LENGTH chosen: {MAX_LENGTH}")

# 4.3 Spot-check
for i, lbl in enumerate(LABELS):
    sample = df_train[df_train["label"] == i]["text"].sample(2, random_state=SEED).tolist()
    print(f"\n[{lbl}] {sample[0][:100]}")


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

ds_tok = ds.map(tokenize, batched=True)
collator = DataCollatorWithPadding(tokenizer=tokenizer)
print("Tokenized splits:", ds_tok)


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }


In [ ]:
args = TrainingArguments(
    output_dir       = "./model2_emotion",
    num_train_epochs = 4,
    per_device_train_batch_size = 32,
    per_device_eval_batch_size  = 64,
    learning_rate    = 2e-5,
    weight_decay     = 0.01,
    warmup_ratio     = 0.1,
    eval_strategy    = "epoch",
    save_strategy    = "epoch",
    load_best_model_at_end  = True,
    metric_for_best_model   = "f1_macro",
    greater_is_better       = True,
    save_total_limit = 2,
    logging_steps    = 50,
    fp16             = torch.cuda.is_available(),
    report_to        = "none",
    seed             = SEED,
)

trainer = Trainer(
    model           = model,
    args            = args,
    train_dataset   = ds_tok["train"],
    eval_dataset    = ds_tok["validation"],
    data_collator   = collator,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=2)],
)
trainer.train()


In [ ]:
res    = trainer.predict(ds_tok["test"])
preds  = np.argmax(res.predictions, axis=-1)
labels = res.label_ids

acc = accuracy_score(labels, preds)
f1  = f1_score(labels, preds, average="macro")
gate = "PASS" if acc >= ACCURACY_GATE else "FAIL"

print(f"\n=== MODEL 2 — TEST RESULTS ===")
print(f"Accuracy : {acc:.4f}   80% gate → {gate}")
print(f"Macro F1 : {f1:.4f}")
print(classification_report(labels, preds, target_names=LABELS))

cm = confusion_matrix(labels, preds)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=LABELS, yticklabels=LABELS)
plt.title("Model 2 — Confusion Matrix (Test)")
plt.ylabel("True"); plt.xlabel("Predicted")
plt.tight_layout()
plt.savefig("model2_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

summary = {
    "model": "Model 2 - Emotion Classifier (RoBERTa, dair-ai/emotion 6-class)",
    "base": MODEL_NAME, "dataset": "dair-ai/emotion",
    "test_accuracy": round(acc, 4), "test_f1_macro": round(f1, 4),
    "gate": gate, "labels": LABELS,
    "n_train": len(ds["train"]), "n_val": len(ds["validation"]),
    "n_test": len(ds["test"]),
}
with open("model2_results_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))


In [ ]:
model_card = f"""---
language: en
license: apache-2.0
tags:
  - text-classification
  - emotion
  - roberta
datasets:
  - dair-ai/emotion
metrics:
  - accuracy
  - f1
pipeline_tag: text-classification
---

# MindLens Emotion Classifier (RoBERTa, dair-ai/emotion)

Fine-tuned `{MODEL_NAME}` for 6-class emotion classification
as part of the MindLens mental-health support system.

**Classes:** {", ".join(LABELS)}

## Results (held-out test set, n={len(ds["test"])})
| Metric | Score |
|---|---|
| Accuracy | {acc:.4f} |
| Macro F1 | {f1:.4f} |
| 80% gate | {gate} |

## Dataset
[dair-ai/emotion](https://huggingface.co/datasets/dair-ai/emotion) —
~16k English Twitter messages labelled with 6 basic emotions.
Train: {len(ds["train"]):,} | Val: {len(ds["validation"]):,} | Test: {len(ds["test"]):,}

## Usage
```python
from transformers import pipeline
clf = pipeline("text-classification",
               model="AmiruMallawarachchi/mindlens-emotion-classifier")
clf("I feel completely overwhelmed and hopeless")
```

## Limitations
Twitter-domain training data. One signal among five in the MindLens
orchestrator — not a clinical mood assessment.
"""

api = HfApi(token=HF_TOKEN)
create_repo(HF_REPO_ID, token=HF_TOKEN, exist_ok=True)
trainer.save_model("./model2_final")
tokenizer.save_pretrained("./model2_final")
with open("./model2_final/README.md", "w") as f:
    f.write(model_card)
api.upload_folder(folder_path="./model2_final",
                  repo_id=HF_REPO_ID, repo_type="model")
print(f"\nPushed → https://huggingface.co/{HF_REPO_ID}")
